# Silver Cleaning

## Imports

In [1]:
from pyspark.sql.functions import col, trim, substring, to_date, when, sum as spark_sum, current_timestamp, lit

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 3, Finished, Available, Finished, False)

## Read Bronze Table

In [2]:
bronze_df = spark.sql("SELECT * FROM bronze.complaints")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 4, Finished, Available, Finished, False)

In [3]:
display(bronze_df.limit(10))

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a721521-7353-4fd2-a2c0-e67ef9c5a69e)

In [4]:
bronze_df.printSchema()

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 6, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



## Remove Duplicate Rows

In [5]:
rows_before_dedup = bronze_df.count()

silver_deduped_df = bronze_df.dropDuplicates()

rows_after_dedup = silver_deduped_df.count()

duplicate_rows_removed = rows_before_dedup - rows_after_dedup
duplicate_rows_removed_pct = (duplicate_rows_removed / rows_before_dedup) * 100

print(f"Rows before deduplication: {rows_before_dedup:,}")
print(f"Rows after deduplication: {rows_after_dedup:,}")
print(f"Rows removed: {duplicate_rows_removed:,}")
print(f"Percentage removed: {duplicate_rows_removed_pct:.6f}%")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 7, Finished, Available, Finished, False)

Rows before deduplication: 16,294,931
Rows after deduplication: 16,294,807
Rows removed: 124
Percentage removed: 0.000761%


## Validate Required Fields

In [6]:
missing_required_fields_condition = (
    col("complaint_id").isNull() |
    (trim(col("complaint_id")) == "") |
    col("date_received").isNull() |
    (trim(col("date_received")) == "") |
    col("product").isNull() |
    (trim(col("product")) == "") |
    col("issue").isNull() |
    (trim(col("issue")) == "") |
    col("company").isNull() |
    (trim(col("company")) == "")
)

missing_required_fields_df = silver_deduped_df.filter(missing_required_fields_condition)
valid_required_fields_df = silver_deduped_df.filter(~missing_required_fields_condition)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 8, Finished, Available, Finished, False)

In [7]:
missing_required_fields_count = missing_required_fields_df.count()
valid_required_fields_count = valid_required_fields_df.count()

print(f"Rows missing required fields: {missing_required_fields_count:,}")
print(f"Rows having required fields: {valid_required_fields_count:,}")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 9, Finished, Available, Finished, False)

Rows missing required fields: 8,505
Rows having required fields: 16,286,302


In [8]:
display(
    missing_required_fields_df.select(
        "complaint_id",
        "date_received",
        "product",
        "issue",
        "company"
    ).limit(20)
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57cb295a-3071-4da1-bbab-f958770a0d59)

- Rows missing complaint_id, date_received, product, issue, or company were separated from the main Silver DataFrame.
- These records will be kept for quarantine review instead of being dropped.

## Parse Date Fields

In [9]:
silver_dates_df = (
    valid_required_fields_df
    .withColumn(
        "date_received_clean",
        to_date(substring(col("date_received"), 1, 10), "yyyy-MM-dd")
    )
    .withColumn(
        "date_sent_to_company_clean",
        to_date(substring(col("date_sent_to_company"), 1, 10), "yyyy-MM-dd")
    )
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 11, Finished, Available, Finished, False)

In [10]:
display(
    silver_dates_df.select(
        "date_received",
        "date_received_clean",
        "date_sent_to_company",
        "date_sent_to_company_clean"
    ).limit(20)
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46560042-5d74-4836-930c-3e02addfdf0b)

## Validate Date Fields

In [11]:
received_date_missing = (
    col("date_received").isNull()
    | (trim(col("date_received")) == "")
)

received_date_failed_to_parse = (
    col("date_received").isNotNull()
    & (trim(col("date_received")) != "")
    & col("date_received_clean").isNull()
)

sent_date_missing = (
    col("date_sent_to_company").isNull()
    | (trim(col("date_sent_to_company")) == "")
)

sent_date_failed_to_parse = (
    col("date_sent_to_company").isNotNull()
    & (trim(col("date_sent_to_company")) != "")
    & col("date_sent_to_company_clean").isNull()
)

sent_date_before_received_date = (
    col("date_received_clean").isNotNull()
    & col("date_sent_to_company_clean").isNotNull()
    & (col("date_sent_to_company_clean") < col("date_received_clean"))
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 13, Finished, Available, Finished, False)

In [12]:
invalid_date_condition = (
    received_date_missing
    | received_date_failed_to_parse
    | sent_date_failed_to_parse
    | sent_date_before_received_date
)

invalid_date_df = silver_dates_df.filter(invalid_date_condition)

valid_date_df = silver_dates_df.filter(~invalid_date_condition)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 14, Finished, Available, Finished, False)

In [13]:
display(
    silver_dates_df.agg(
        spark_sum(when(received_date_missing, 1).otherwise(0)).alias("missing_date_received"),
        spark_sum(when(received_date_failed_to_parse, 1).otherwise(0)).alias("received_date_failed_to_parse"),
        spark_sum(when(sent_date_missing, 1).otherwise(0)).alias("missing_date_sent_to_company"),
        spark_sum(when(sent_date_failed_to_parse, 1).otherwise(0)).alias("sent_date_failed_to_parse"),
        spark_sum(when(sent_date_before_received_date, 1).otherwise(0)).alias("sent_date_before_received_date")
    )
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3fe8ebfc-ce88-48b6-8431-4478622045b1)

In [14]:
missing_sent_date_count = silver_dates_df.filter(sent_date_missing).count()

invalid_date_count = invalid_date_df.count()
valid_date_count = valid_date_df.count()

date_related_findings_count = missing_sent_date_count + invalid_date_count

print(f"Rows with missing date_sent_to_company: {missing_sent_date_count:,}")
print(f"Rows with invalid date logic: {invalid_date_count:,}")
print(f"Total date related findings: {date_related_findings_count:,}")
print(f"Rows with valid dates: {valid_date_count:,}")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 16, Finished, Available, Finished, False)

Rows with missing date_sent_to_company: 26,215
Rows with invalid date logic: 7,050
Total date related findings: 33,265
Rows with valid dates: 16,279,252


- Some records did not have a date_sent_to_company value, but I kept them because they can still be used for complaint analysis.
- I separated the records where the sent-to-company date was earlier than the received date, since that date order does not make sense.

## Prepare Final Silver DataFrame

In [15]:
silver_complaints_df = (
    valid_date_df
    .withColumn("silver_processed_timestamp", current_timestamp())
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 17, Finished, Available, Finished, False)

In [16]:
display(silver_complaints_df.limit(10))

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5acd0f93-6a7e-45a5-bda4-0bb828320138)

In [17]:
silver_complaints_df.printSchema()

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 19, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- date_received_clean: date (nullable = true)
 |-- date_sent_to_company_clean: date (nullable = true)
 |-- silver_processed_timestamp: 

## Prepare Quarantine DataFrame

In [18]:
missing_required_quarantine_df = (
    missing_required_fields_df
    .withColumn("quarantine_reason", lit("missing_required_field"))
)

invalid_date_quarantine_df = (
    invalid_date_df
    .withColumn("quarantine_reason", lit("date_quality_issue"))
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 20, Finished, Available, Finished, False)

In [19]:
quarantine_complaints_df = missing_required_quarantine_df.unionByName(
    invalid_date_quarantine_df,
    allowMissingColumns=True
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 21, Finished, Available, Finished, False)

In [20]:
display(
    quarantine_complaints_df.select(
        "complaint_id",
        "date_received",
        "date_received_clean",
        "date_sent_to_company",
        "date_sent_to_company_clean",
        "product",
        "issue",
        "company",
        "quarantine_reason"
    ).limit(20)
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce78db44-3f46-4ca6-b361-530abc8a3ae4)

In [21]:
print(f"Rows in silver_complaints: {silver_complaints_df.count():,}")
print(f"Rows in quarantine_complaints: {quarantine_complaints_df.count():,}")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 23, Finished, Available, Finished, False)

Rows in silver_complaints: 16,279,252
Rows in quarantine_complaints: 15,555


## Create Output Schemas

In [22]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS quarantine")

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 24, Finished, Available, Finished, False)

DataFrame[]

## Write Silver Tables

In [23]:
(
    silver_complaints_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.complaints")
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 25, Finished, Available, Finished, False)

In [24]:
(
    quarantine_complaints_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("quarantine.complaints")
)

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 26, Finished, Available, Finished, False)

In [25]:
display(spark.sql("SELECT * FROM silver.complaints LIMIT 10"))

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5dcc66d-82d8-4a31-967e-7010baabe683)

In [26]:
display(spark.sql("SELECT * FROM quarantine.complaints LIMIT 10"))

StatementMeta(, cc154f79-8ed2-437d-a70d-e103c5458e9a, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdc7edff-36c5-4806-896f-4014bee0e13d)